In [2]:
import os
import sklearn
import uproot
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import roc_curve, auc
import xgboost as xgb
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve
# import optuna
# import optuna.visualization as vis
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import joblib
import shap
import pandas as pd
import plotly.express as px
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.nn.functional as F

base = "/home/champson/data/fastframes"
# -----------------------------
# lq masses (1.0 yukawa only)
# -----------------------------
signal_file_1 = [ # 1500 GeV
    f"{base}/mc20a/545824.root",
    f"{base}/mc20d/545824.root",
    f"{base}/mc20e/545824.root",
]
signal_file_2 = [ # 2000 GeV
    f"{base}/mc20a/545825.root",
    f"{base}/mc20d/545825.root",
    f"{base}/mc20e/545825.root",
]
signal_file_3 = [ # 2500 GeV
    f"{base}/mc20a/545826.root",
    f"{base}/mc20d/545826.root",
    f"{base}/mc20e/545826.root",
]
signal_file_4 = [ # 1600 GeV
    f"{base}/mc20a/567831.root",
    f"{base}/mc20d/567831.root",
    f"{base}/mc20e/567831.root",
]
signal_file_5 = [ # 1700 GeV
    f"{base}/mc20a/567832.root",
    f"{base}/mc20d/567832.root",
    f"{base}/mc20e/567832.root",
]
signal_file_6 = [ # 1800 GeV
    f"{base}/mc20a/567833.root",
    f"{base}/mc20d/567833.root",
    f"{base}/mc20e/567833.root",
]
signal_file_7 = [ # 1900 GeV
    f"{base}/mc20a/567834.root",
    f"{base}/mc20d/567834.root",
    f"{base}/mc20e/567834.root",
]
signal_file_8 = [ # 2100 GeV
    f"{base}/mc20a/567835.root",
    f"{base}/mc20d/567835.root",
    f"{base}/mc20e/567835.root",
]
signal_file_9 = [ # 2200 GeV
    f"{base}/mc20a/567836.root",
    f"{base}/mc20d/567836.root",
    f"{base}/mc20e/567836.root",
]
signal_file_10 = [ # 2300 GeV
    f"{base}/mc20a/567837.root",
    f"{base}/mc20d/567837.root",
    f"{base}/mc20e/567837.root",
]
signal_file_11 = [ # 2400 GeV
    f"{base}/mc20a/567838.root",
    f"{base}/mc20d/567838.root",
    f"{base}/mc20e/567838.root",
]

# -----------------------------
# backgrounds
# -----------------------------
ttH = [
    f"{base}/mc20a/346343.root",
    f"{base}/mc20d/346343.root",
    f"{base}/mc20e/346343.root",
    
    f"{base}/mc20a/346344.root",
    f"{base}/mc20d/346344.root",
    f"{base}/mc20e/346344.root",
    
    f"{base}/mc20a/346345.root",
    f"{base}/mc20d/346345.root",
    f"{base}/mc20e/346345.root",
]

ttW = [
    f"{base}/mc20a/701261.root",
    f"{base}/mc20d/701261.root",
    f"{base}/mc20e/701261.root",
    
    f"{base}/mc20a/701262.root",
    f"{base}/mc20d/701262.root",
    f"{base}/mc20e/701262.root",
]

ttZ = [
    f"{base}/mc20a/504330.root",
    f"{base}/mc20d/504330.root",
    f"{base}/mc20e/504330.root",
    
    f"{base}/mc20a/504334.root",
    f"{base}/mc20d/504334.root",
    f"{base}/mc20e/504334.root",
    
    f"{base}/mc20a/504342.root",
    f"{base}/mc20d/504342.root",
    f"{base}/mc20e/504342.root",
]

ttbar = [
    f"{base}/mc20a/410470.root",
    f"{base}/mc20d/410470.root",
    f"{base}/mc20e/410470.root",
]


vv = [
    f"{base}/mc20a/701000.root",
    f"{base}/mc20d/701000.root",
    f"{base}/mc20e/701000.root",
    
    f"{base}/mc20a/701005.root",
    f"{base}/mc20d/701005.root",
    f"{base}/mc20e/701005.root",
    
    f"{base}/mc20a/701010.root",
    f"{base}/mc20d/701010.root",
    f"{base}/mc20e/701010.root",

    f"{base}/mc20a/701015.root",
    f"{base}/mc20d/701015.root",
    f"{base}/mc20e/701015.root",
    
    f"{base}/mc20a/701020.root",
    f"{base}/mc20d/701020.root",
    f"{base}/mc20e/701020.root",
    
    f"{base}/mc20a/701025.root",
    f"{base}/mc20d/701025.root",
    f"{base}/mc20e/701025.root",
    
    f"{base}/mc20a/701030.root",
    f"{base}/mc20d/701030.root",
    f"{base}/mc20e/701030.root",
    
    f"{base}/mc20a/701035.root",
    f"{base}/mc20d/701035.root",
    f"{base}/mc20e/701035.root",
    
    f"{base}/mc20a/701055.root",
    f"{base}/mc20d/701055.root",
    f"{base}/mc20e/701055.root",
    
    f"{base}/mc20a/701085.root",
    f"{base}/mc20d/701085.root",
    f"{base}/mc20e/701085.root",
    
    f"{base}/mc20a/701090.root",
    f"{base}/mc20d/701090.root",
    f"{base}/mc20e/701090.root",
    
    f"{base}/mc20a/701105.root",
    f"{base}/mc20d/701105.root",
    f"{base}/mc20e/701105.root",
    
    f"{base}/mc20a/701115.root",
    f"{base}/mc20d/701115.root",
    f"{base}/mc20e/701115.root",
    
    f"{base}/mc20a/701120.root",
    f"{base}/mc20d/701120.root",
    f"{base}/mc20e/701120.root",
    
    f"{base}/mc20a/701125.root",
    f"{base}/mc20d/701125.root",
    f"{base}/mc20e/701125.root",
]

vvv = [
    f"{base}/mc20a/364242.root",
    f"{base}/mc20d/364242.root",
    f"{base}/mc20e/364242.root",
    
    f"{base}/mc20a/364243.root",
    f"{base}/mc20d/364243.root",
    f"{base}/mc20e/364243.root",
    
    f"{base}/mc20a/364244.root",
    f"{base}/mc20d/364244.root",
    f"{base}/mc20e/364244.root",
    
    f"{base}/mc20a/364245.root",
    f"{base}/mc20d/364245.root",
    f"{base}/mc20e/364245.root",
    
    f"{base}/mc20a/364246.root",
    f"{base}/mc20d/364246.root",
    f"{base}/mc20e/364246.root",
    
    f"{base}/mc20a/364247.root",
    f"{base}/mc20d/364247.root",
    f"{base}/mc20e/364247.root",
    
    f"{base}/mc20a/364248.root",
    f"{base}/mc20d/364248.root",
    f"{base}/mc20e/364248.root",
    
    f"{base}/mc20a/364249.root",
    f"{base}/mc20d/364249.root",
    f"{base}/mc20e/364249.root",
]


vh = [
    f"{base}/mc20a/346645.root",
    f"{base}/mc20d/346645.root",
    f"{base}/mc20e/346645.root",
    
    f"{base}/mc20a/346646.root",
    f"{base}/mc20d/346646.root",
    f"{base}/mc20e/346646.root",
]


vgamma = [
    f"{base}/mc20a/700398.root",
    f"{base}/mc20d/700398.root",
    f"{base}/mc20e/700398.root",
    f"{base}/mc20a/700399.root",
    f"{base}/mc20d/700399.root",
    f"{base}/mc20e/700399.root",
    f"{base}/mc20a/700400.root",
    f"{base}/mc20d/700400.root",
    f"{base}/mc20e/700400.root",
#    f"{base}/mc20a/700401.root",
#    f"{base}/mc20d/700401.root",
#    f"{base}/mc20e/700401.root",
    f"{base}/mc20a/700402.root",
    f"{base}/mc20d/700402.root",
    f"{base}/mc20e/700402.root",
    f"{base}/mc20a/700403.root",
    f"{base}/mc20d/700403.root",
    f"{base}/mc20e/700403.root",
    f"{base}/mc20a/700404.root",
    f"{base}/mc20d/700404.root",
    f"{base}/mc20e/700404.root",
]

wjets = [
    f"{base}/mc20a/700338.root",
    f"{base}/mc20d/700338.root",
    f"{base}/mc20e/700338.root",
    f"{base}/mc20a/700341.root",
    f"{base}/mc20d/700341.root",
    f"{base}/mc20e/700341.root",
]

zjets = [
    f"{base}/mc20a/700320.root",
    f"{base}/mc20d/700320.root",
    f"{base}/mc20e/700320.root",
    f"{base}/mc20a/700321.root",
    f"{base}/mc20d/700321.root",
    f"{base}/mc20e/700321.root",
#    f"{base}/mc20a/700322.root",
#    f"{base}/mc20d/700322.root",
#    f"{base}/mc20e/700322.root",
    f"{base}/mc20a/700323.root",
    f"{base}/mc20d/700323.root",
    f"{base}/mc20e/700323.root",
    f"{base}/mc20a/700324.root",
    f"{base}/mc20d/700324.root",
    f"{base}/mc20e/700324.root",
    f"{base}/mc20a/700325.root",
    f"{base}/mc20d/700325.root",
    f"{base}/mc20e/700325.root",
]


# Rare tops


ttHH = [
    f"{base}/mc20a/500460.root",
    f"{base}/mc20d/500460.root",
    f"{base}/mc20e/500460.root",
]

ttWH = [
    f"{base}/mc20a/500461.root",
    f"{base}/mc20d/500461.root",
    f"{base}/mc20e/500461.root",
]

ttWW = [
    f"{base}/mc20a/410081.root",
    f"{base}/mc20d/410081.root",
    f"{base}/mc20e/410081.root",
]

ttWZ = [
    f"{base}/mc20a/500463.root",
    f"{base}/mc20d/500463.root",
    f"{base}/mc20e/500463.root",
]

ttZZ = [
    f"{base}/mc20a/500462.root",
    f"{base}/mc20d/500462.root",
    f"{base}/mc20e/500462.root",
]

ttgamma = [
    f"{base}/mc20a/500462.root",
    f"{base}/mc20d/500462.root",
    f"{base}/mc20e/500462.root",
    
    f"{base}/mc20a/504554.root",
    f"{base}/mc20d/504554.root",
    f"{base}/mc20e/504554.root",
]

# Other rare processes
tZ = [
    f"{base}/mc20a/410560.root",
    f"{base}/mc20d/410560.root",
    f"{base}/mc20e/410560.root",
]

WtZ = [
    f"{base}/mc20a/410408.root",
    f"{base}/mc20d/410408.root",
    f"{base}/mc20e/410408.root",
]

fourTop = [
    f"{base}/mc20a/412043.root",
    f"{base}/mc20d/412043.root",
    f"{base}/mc20e/412043.root",
]


singleTop = [
    f"{base}/mc20a/410644.root",
    f"{base}/mc20d/410644.root",
    f"{base}/mc20e/410644.root",
    
    f"{base}/mc20a/410645.root",
    f"{base}/mc20d/410645.root",
    f"{base}/mc20e/410645.root",
    
    f"{base}/mc20a/410654.root",
    f"{base}/mc20d/410654.root",
    f"{base}/mc20e/410654.root",
    
    f"{base}/mc20a/410655.root",
    f"{base}/mc20d/410655.root",
    f"{base}/mc20e/410655.root",
    
    f"{base}/mc20a/410659.root",
    f"{base}/mc20d/410659.root",
    f"{base}/mc20e/410659.root",
]


threeTop = [
    f"{base}/mc20a/304014.root",
    f"{base}/mc20d/304014.root",
    f"{base}/mc20e/304014.root",
]



background_files = ttH + ttW + ttZ + ttbar + vv + vvv + vh + vgamma + wjets + zjets + ttHH + ttWH + ttWW + ttWZ + ttZZ + ttgamma + tZ + WtZ + fourTop + singleTop + threeTop
signal_files_all = signal_file_1 + signal_file_2 + signal_file_3 + signal_file_4 + signal_file_5 + signal_file_6 + signal_file_7 + signal_file_8 + signal_file_9 + signal_file_10 + signal_file_11



tree_name = "reco"

selected_features = [
    "taus_pt_0_NOSYS",
    "m_eff_NOSYS",
    "HT_leptons_NOSYS",
    "jets_n_NOSYS",
    "pT_balance_lep_tau_MET_NOSYS",
    "min_DeltaR_tau0_SSlepton_NOSYS",
    "b0_pt_NOSYS",
    "MLepMet_NOSYS",
    "HT_NOSYS",
    "jet_0_1_phi_diff_cos_NOSYS",
    "top_reco_mass_l0b0_NOSYS",
    "dEta_maxMjj_frwdjet_NOSYS",
    "lep_deltaz0sinTheta_0_NOSYS",
    "DeltaR_min_lep_jet_fwd_NOSYS",
    "minDeltaR_LJ_1_NOSYS",
    "lep_d0sig_1_NOSYS",
    "nbJets77_NOSYS",
    "min_AbsDeltaPhi_bjet_MET_NOSYS",
    "EtaZeppen_tau_NOSYS",
    "reco_Hplus_visH_hadW_mass_NOSYS",
    "min_DeltaR_tau_bjet_NOSYS",
    "leps_charge_0_NOSYS",
    "pT_vector_balance_NOSYS",
    "had_W_jj_mass_NOSYS",
    "lep1_tau_Phi_diff_cos_NOSYS",
    "MET_centrality_NOSYS",
    "taus_RNNJetScoreSigTrans_0_NOSYS",
    "leps_pt_1_NOSYS",
    "lep_d0sig_0_NOSYS",
    "DeltaPhi_SSLeptonSystem_Tau_NOSYS",
    "DeltaR_L0_L1_NOSYS",
    "DeltaPhi_T0_MET_NOSYS",
    "lep_Z0SinTheta_1_NOSYS",
    "min_chi2_top_reco_NOSYS",
]



signal_by_mass = {
    1500:  signal_file_1,
    2000: signal_file_2,
    2500:  signal_file_3,
    1600:  signal_file_4,
    1700:  signal_file_5,
    1800:  signal_file_6,
    1900:  signal_file_7,
    2100:  signal_file_8,
    2200: signal_file_9,
    2300: signal_file_10,
    2400: signal_file_11,
}

pnn_features = list(dict.fromkeys(selected_features))
# The pNN model will see these features PLUS log(m) appended as the last column.
# Total input width = len(pnn_features) + 1
 
print(f"pNN feature set: {len(pnn_features)} physics features + 1 mass column")


cut_expr = "(taus_n_NOSYS == 1) and (jets_n_NOSYS >= 4) and (nbJets77_NOSYS >= 1) and (abs(Mll01_NOSYS/1.0e3 - 91.2) > 10.0) and (Mll01_NOSYS/1.0e3 > 12.0) and (leps_pt_0_NOSYS/1.0e3 > 25.0) and (leps_pt_1_NOSYS/1.0e3 > 10.0)"

pNN feature set: 34 physics features + 1 mass column


In [3]:
def extract_feature(x):
    if isinstance(x, (list, np.ndarray)):
        return float(np.max(x)) if len(x) > 0 else np.nan
    return float(x)
 
 
def load_root_files(file_paths, tree_name, features, cut_expr):
    """Load ROOT files and return a plain feature matrix (no mass column)."""
    chunks = []
    for fp in file_paths:
        print(f"  Loading: {fp}")
        with uproot.open(fp) as f:
            df = f[tree_name].arrays(features, library="pd", cut=cut_expr)
        for feat in features:
            df[feat] = df[feat].apply(extract_feature)
        df = df.fillna(0)
        chunks.append(df[features].values.astype(np.float32))
    if not chunks:
        return np.empty((0, len(features)), dtype=np.float32)
    return np.concatenate(chunks, axis=0)
 
 
def append_log_mass(X, mass_value):
    """Append log(mass_value) as a constant last column to every row."""
    log_m = np.full((len(X), 1), np.log(float(mass_value)), dtype=np.float32)
    return np.concatenate([X, log_m], axis=1)
 
 
def find_best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = (2 * precision[:-1] * recall[:-1]
                 / (precision[:-1] + recall[:-1] + 1e-12))
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx], f1_scores[best_idx]
 
 
def compute_conf_matrix(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return cm, tn, fp, fn, tp
 
 
def save_prf1_vs_threshold_plot(y_true, y_score, title, out_path, n_thr=1000):
    y_true  = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    thresholds = np.linspace(0.0, 1.0, n_thr + 1)
    precisions = np.zeros_like(thresholds)
    recalls    = np.zeros_like(thresholds)
    f1s        = np.zeros_like(thresholds)
    sigs       = np.zeros_like(thresholds)
    for i, t in enumerate(thresholds):
        y_pred = (y_score >= t).astype(int)
        precisions[i] = precision_score(y_true, y_pred, zero_division=0)
        recalls[i]    = recall_score(y_true, y_pred, zero_division=0)
        f1s[i]        = f1_score(y_true, y_pred, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        sigs[i] = tp / np.sqrt(fp) if fp > 0 else 0.0
    best_f1_idx  = np.argmax(f1s)
    best_sig_idx = np.argmax(sigs)
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(thresholds, precisions, "b--", label="Precision")
    ax1.plot(thresholds, recalls,    "g-",  label="Recall")
    ax1.plot(thresholds, f1s,        "r-",  label="F1-Score")
    ax1.axvline(thresholds[best_f1_idx], color="purple", linestyle="--",
                label=f"Best F1 thr ({thresholds[best_f1_idx]:.2f})")
    ax1.set_xlabel("Threshold"); ax1.set_ylabel("Score"); ax1.set_ylim(0, 1.02)
    ax1.grid(True, alpha=0.3)
    ax2 = ax1.twinx()
    ax2.plot(thresholds, sigs, "-", label="TP/sqrt(FP)")
    ax2.axvline(thresholds[best_sig_idx], color="black", linestyle=":",
                label=f"Best S/sqrt(B) thr ({thresholds[best_sig_idx]:.2f})")
    ax2.set_ylabel("TP/sqrt(FP)")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower center")
    plt.title(title); plt.tight_layout(); plt.savefig(out_path, dpi=200); plt.close()
    return thresholds[best_f1_idx], f1s[best_f1_idx], thresholds[best_sig_idx], sigs[best_sig_idx]

In [5]:
print("\n=== Loading background ===")
X_bkg_raw = load_root_files(background_files, tree_name, pnn_features, cut_expr)
print(f"Background events: {len(X_bkg_raw)}")


=== Loading background ===
  Loading: /home/champson/data/fastframes/mc20a/346343.root


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
print("\n=== Loading signal by mass ===")
mass_grid  = np.array(sorted(signal_by_mass.keys()), dtype=np.float32)
X_sig_list = []
for m, files in sorted(signal_by_mass.items()):
    X = load_root_files(files, tree_name, pnn_features)
    X_sig_list.append((m, X))
    print(f"  m={m:5d} GeV: {len(X)} events")

In [ ]:
X_sig_raw_all  = np.concatenate([X for _, X in X_sig_list], axis=0)
m_sig_true_all = np.concatenate([np.full(len(X), m) for m, X in X_sig_list])